In [1]:
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:1234/v1",api_key="lm-studio")
EMBED_MODEL = "text-embedding-bge-m3"

# Idle TTL 60min
60分使わないと、LM Studioが自動モデルをおろしてVRAMを開ける設定。降ろされても、Just-in-time loadingがONなら次に読んだ時勝手印ロードし直してくれる。

In [7]:
response = client.embeddings.create(
    model=EMBED_MODEL,
    input="今日は朝から雨が降っていたので、傘を持って学校に行きました。授業ではRAGについて学び、文章を数字に変換する埋め込みという仕組みを知りました。",
)

vec = response.data[0].embedding

- 文章の意味を1024個の「意味の軸」で表しているイメージ
- この軸=動物といったようにはっきりきまっているわけではなく、モデルが学習してその中で作った軸
- この1024個は文章の長さで変わらない

In [8]:
print("数字の個数:", len(vec))
print("最初の5個:", vec[:5])

数字の個数: 1024
最初の5個: [-0.0031158183701336384, 0.029353952035307884, 0.00985335186123848, -0.00962953083217144, -0.027116788551211357]


In [14]:
import numpy as np

def embed(text):
    response = client.embeddings.create(model=EMBED_MODEL, input=text)
    return np.array(response.data[0].embedding)

def cos_sim(a,b):
    return np.dot(a,b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [15]:
base = embed("犬が好き")

texts = [
    "わんこ大好き",
    "猫が好き",
    "犬が嫌い",
    "株価が下がった",
]

for t in texts:
    print(f"{cos_sim(base, embed(t)):.3f} {t}")

0.788 わんこ大好き
0.830 猫が好き
0.799 犬が嫌い
0.477 株価が下がった


# コサイン類似度
- 0 < x < 1で0に近いほど関係がなく、1に近くなるほどほぼ意味が同じになる
- 変数baseに近い意味で検証したのが上記の結果。猫が好きが文体と「好き」という部分が似ているためもっとも高くなっている。またわんこ大好きは意味こそ同じだが、処理の仕方によってはかわることがわかる
- 文の形、使っている言葉、否定は苦手、言い方が違うなどの要因に作用されることがわかる
- よって同じ話題について書いてある部分を探すのは得意だが、正しい答えがどうかまでは判断しきれない
- 埋め込み：関係ありそうな部分をざっくり拾う、AI：それを読んで、正しく理解して答える
- また補足として「株価が下がった」という文は0にならなったのは日本語で短い文という共通点があるからだ

In [16]:
base = embed("犬は何年くらい生きるの？")

texts = [
    "犬の平均寿命は10~15年ほどです",
    "猫の平均寿命は12~18年ほどです",
    "犬は嗅覚がとても優れています",
    "日経平均株価が下落しました",
]

for t in texts:
    print(f"{cos_sim(base, embed(t)):.3f} {t}")

0.786 犬の平均寿命は10~15年ほどです
0.646 猫の平均寿命は12~18年ほどです
0.574 犬は嗅覚がとても優れています
0.338 日経平均株価が下落しました


# まとめ
- 文章は長さに関係なく1024個の数字になるため比べられる
- 似ているかどうかは**コサイン類似度**で計算する
- 単語が違っても**意味が近ければ近く**なる
- でも否定や正しさまでは判断しない→AIの仕事
- 数値そのものより順位が重要